## Diploma Thesis: Multi-Block Replacement

The objective of this notebook is to extend work on introduction notebook. Here we will focus on create an MVP methodology on how to replace multiple blocks inside an input transformer model.  
The key difference is that in the case of multi-block replacements and error propagation occurs across the model (Grafting paper). To ensure efficiency we utilize again the calibration data  
for local scope working as well as knowledge distillation for fine-tune retraining.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint
from typing import Literal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
project_pwd = Path().cwd().parents[1]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

In [3]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

from scripts.utils import *
from configs.intro_config import *

Model config - extended to multi-block workflow

In [4]:
mcfg = MCFG()
torch.manual_seed(mcfg.seed)

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

### Workflow

1) Load model and tokenizer
2) make data loader
3) collect blocks i/o pairs (per block)
4) evaluate block importance (per block)
5) select replacement strategy
6) apply replacement strategy on blocks
7) fine-tuning based on replacement strategy
8) eval

___

model utils

In [3]:
def layer_idx_to_mlp_path(model, layer_idx):
    paths = list_mlp_paths(model)
    expected = f"model.layers.{layer_idx}.mlp"

    if expected in paths:
        return expected
    
    matches = []
    for p in paths:
        if p.endswith(".mlp") and f".{layer_idx}.mlp" in p:
            matches.append(p)

    if not matches:
        raise ValueError("bad MLP path provided.")
    return matches[0]

block scoring

In [11]:
def bi_cosine_score(X, Y):
    cos = F.cosine_similarity(X, Y, dim=-1, eps=1e-8)
    score = 1.0 - cos
    score = score.mean().item()
    return float(score)


def compute_bi_scores(model, layer_idxs, loader, cfg, device):
    scores = {}
    for idx in layer_idxs:
        idx = int(idx)
        path = layer_idx_to_mlp_path(model, idx)
        X, Y = collect_block_io(model, path, cfg.num_calib_batches, device)
        scores[idx] = bi_cosine_score(X, Y)

    return scores

vyber operatora

In [17]:
def fit_replacement_operator(operator_name, X, Y, hidden_size, device, cfg):
    if operator_name == "linear":
        return fit_linear_replacement(X, Y, hidden_size, device, cfg)
    
    raise ValueError(f"Unsupported operator, operator={operator_name}")

recovery funkcie

In [19]:
def global_recovery_finetunning(model, calib_loader, cfg, device):
    pass

def local_recovery_finetunning(model, calib_loader, cfg, device):
    pass

replacement strategy

In [21]:
def run_one_shot_replacement(model, target_layer_idxs, calib_loader, eval_loader):
    pass

def run_iterative_replacement(model, target_layer_idxs, calib_loader, eval_loader):
    pass

___

### Testovanie experimentov

Otazky:

- Ako urcit dostatocne dobre k? Zatial to bolo manualne
- Treba robit selekciu operatorov rucne alebo nejakym rozumnym sposobom vieme automatizovat a spravit vztah medzi BI score a operatorom?